# Entrenamiento de Modelo Clasificador (Enrutador)

Este notebook configura y ejecuta el entrenamiento de un modelo de clasificación de secuencias basado en `microsoft/Multilingual-MiniLM-L12-H384`. Su objetivo es categorizar peticiones en lenguaje natural para enrutarlas a diferentes subsistemas (ej. Docker, Red, SysAdmin).

**Funciones principales:**
* **Carga y Preparación:** Lee un dataset local en formato JSONL y divide los datos para entrenamiento y validación.
* **Tokenización:** Procesa el texto y mapea las etiquetas de clase a identificadores numéricos.
* **Entrenamiento Optimizado:** Utiliza aceleración GPU con formato `bfloat16` (óptimo para arquitecturas modernas como RTX serie 40/50) y paraleliza la carga de datos.
* **Evaluación y Exportación:** Calcula la precisión (accuracy) en cada época y guarda el mejor modelo resultante en el directorio local.

**Requisitos de Hardware y Sistema:**
* **GPU:** Tarjeta gráfica NVIDIA con soporte nativo para `bf16` y VRAM suficiente.
* **Sistema Operativo:** Entorno Linux configurado para carga paralela de datos (probado en Fedora Linux 43 con KDE).

**Dependencias requeridas (pip):**
* `transformers`
* `datasets`
* `evaluate`
* `accelerate`
* `scikit-learn`
* `torch`

In [ ]:
# <---------------------------- Instalacion De Depenencias ------------------------------>
!pip install -q transformers datasets evaluate accelerate scikit-learn torch

In [ ]:
# Instalacion de librerias (usar %pip en Jupyter o instalar en el entorno virtual)
# %pip install -q transformers datasets evaluate accelerate scikit-learn torch

import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import evaluate
import numpy as np

# Validar disponibilidad de GPU 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo actual: {device}")
if torch.cuda.is_available():
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")

# Rutas locales
model_name = "microsoft/Multilingual-MiniLM-L12-H384"
output_dir = "./output/minilm-l12-lemon-route" # <-------------- Ruta de salida del modelo

# Configuracion de etiquetas
labels_list = [
    "malbec",      # Docker
    "syrah",       # Red
    "tempranillo", # SysAdmin
    "pinot",       # Busqueda
    "chardonnay",  # Archivos
    "cabernet",    # Remoto
    "gemma",       # General
    "null",        # Fuera de dominio
    "secure"
    # Aqui podemos agregar mas categorias en base a las necesidades que tengamos 
]

label2id = {label: i for i, label in enumerate(labels_list)}
id2label = {i: label for i, label in enumerate(labels_list)}

# Cargar dataset local
dataset_file = "./data/router_final.jsonl" # <------------- Ruta al dataset a usar

if not os.path.exists(dataset_file):
    raise FileNotFoundError(f"No se encontro '{dataset_file}' en el directorio de trabajo.")

dataset = load_dataset("json", data_files=dataset_file)

# Division train/test (90/10)
dataset = dataset["train"].train_test_split(test_size=0.1)

# Tokenizacion
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    # Tokeniza y mapea etiquetas a IDs numericos
    tokenized = tokenizer(examples["text"], truncation=True, padding=False)
    tokenized["labels"] = [label2id[label] for label in examples["label"]]
    return tokenized

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

# Inicializar modelo y mover a GPU
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels_list),
    id2label=id2label,
    label2id=label2id
).to(device)

# Evaluacion
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

# Configuracion de entrenamiento para GPU local
training_args = TrainingArguments(
    output_dir="./resultados",
    learning_rate=5e-5,
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2,
    logging_dir='./logs',
    logging_steps=50,
    bf16=True,                 # Formato de precision optima para RTX serie 40/50
    dataloader_num_workers=4,  # Paraleliza la carga de datos en Linux
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,  # Revertido a 'tokenizer' para compatibilidad con tu version
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

# Iniciar entrenamiento
print("Iniciando entrenamiento...")
trainer.train()

# Guardar en directorio local
print(f"Guardando modelo en: {output_dir}")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print("Proceso completado.")

In [ ]:
import os
import torch
from transformers import pipeline

# Ruta del modelo entrenado en local
model_path = "./output/minilm-l12-lemon-route" # <------------------ Ruta de salida del modelo entrenado anteriormente

# Verificar que el directorio del modelo exista
if not os.path.exists(model_path):
    raise FileNotFoundError(f"No se encontro el modelo en: {model_path}")

# Asignar dispositivo: 0 para GPU, -1 para CPU
device = 0 if torch.cuda.is_available() else -1
device_name = "GPU" if device == 0 else "CPU"
print(f"Cargando modelo desde {model_path} en {device_name}...")

# Inicializar pipeline de clasificacion de texto
classifier = pipeline("text-classification", model=model_path, tokenizer=model_path, device=device)

print("\n--- Inferencia Interactiva ---")
print("Escribe 'salir' para terminar.\n")

# Bucle para ingreso de texto y evaluacion
while True:
    text = input("Entrada: ").strip()

    if text.lower() in ['salir', 'exit', 'quit']:
        break
        
    if not text:
        continue

    # Procesar entrada y obtener la clase con mayor probabilidad
    try:
        prediction = classifier(text)[0]
        label = prediction['label']
        score = prediction['score']

        print(f"-> Categoria: {label}")
        print(f"   Confianza: {score:.2%}\n")

    except Exception as e:
        print(f"Error en inferencia: {e}")